# Generate Location Split Dataset

This notebook generates patient bundles split between two hospitals using a **location strategy**.
Encounters are grouped by location, and different locations are partitioned between hospitals.

## Import Libraries and Load Resources

Load the resources, graphs, and helper functions from the main pipeline.

In [1]:
from pathlib import Path
from strategies import SPLIT_STRATEGIES
from dataset_factory import build_split_dataset_with_sets, get_output_dir_for_split
from persistence import persist_patient_bundles

## Build Graphs and Load Resources

Load the FHIR resources from compressed NDJSON files and build forward/reverse reference graphs.

In [2]:
from graph_builder import build_fhir_graphs

# Load FHIR resources and build graphs
forward_graph, reverse_graph, resources_map = build_fhir_graphs()

Processing: Condition
Processing: Encounter
Processing: Encounter
Processing: Location
Processing: Medication
Processing: MedicationAdministration
Processing: MedicationAdministration
Processing: MedicationDispense
Processing: MedicationRequest
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Organization
Processing: Patient
Processing: Procedure
Processing: Procedure
Processing: Specimen
Processing: Specimen

Graphs built successfully!
Resource types: ['Condition', 'Encounter', 'Location', 'Medication', 'MedicationAdministration', 'MedicationDispense', 'MedicationRequest', 'Observation', 'Organization', 'Patient', 'Procedure', 'Specimen']
Total resources: 887269


## Configure Strategy

Select the location splitting strategy.

In [3]:
# Select strategy and dataset name
strategy = SPLIT_STRATEGIES["location"]
dataset_name = "location"
seed = 42  # Optional: set seed for reproducibility

print(f"Strategy: {strategy.__name__}")
print(f"Dataset name: {dataset_name}")

Strategy: location_split
Dataset name: location


## Build Split Dataset

Use the strategy to split patient encounters between hospital_A and hospital_B.

In [4]:
print("Building split dataset...")
results = build_split_dataset_with_sets(
    strategy=strategy,
    resources_map=resources_map,
    reverse_graph=reverse_graph,
    forward_graph=forward_graph,
    seed=seed
)

print(f"Built datasets for {len(results)} splits")
for split_name, locations in results.items():
    print(f"  {split_name}: {sum(len(bundles) for bundles in locations.values())} patients")

Building split dataset...
Patient Patient/837b0984-f3bf-5076-baf5-b727decb4bea has only one location; falling back to temporal split.
Built datasets for 3 splits
  train: 120 patients
  validation: 40 patients
  test: 40 patients


## Persist Datasets to Disk

Save the split bundles to `input/{dataset_name}/hospital_A` and `input/{dataset_name}/hospital_B`.

In [5]:
print("Persisting bundles...")

total_persisted = 0
for split_name, locations in results.items():
    for location_name, patient_bundles in locations.items():
        if not patient_bundles:  # Skip empty locations
            continue
        
        output_dir = get_output_dir_for_split(
            base_dir=f"input/{dataset_name}",
            split_set=split_name,
            location_name=location_name
        )
        
        persisted = persist_patient_bundles(
            patient_bundles,
            output_dir,
            resources_map
        )
        
        print(f"Saved {persisted} bundles to {output_dir}")
        total_persisted += persisted

print(f"\nTotal bundles persisted: {total_persisted}")

Persisting bundles...
Saved 60 bundles to input\location\train\hospital_a
Saved 60 bundles to input\location\train\hospital_b
Saved 20 bundles to input\location\validation\hospital_a
Saved 20 bundles to input\location\validation\hospital_b
Saved 20 bundles to input\location\test\hospital_a
Saved 20 bundles to input\location\test\hospital_b

Total bundles persisted: 200
